## Read data, convert to json
Gerrychain takes json as input. The geopandas package should be converted first to graph and then a json. We use data on the census block group level.

In [ ]:
import geopandas as gpd
import fiona
from pathlib import Path
from gerrychain import Graph


# Read in geopackage
gpkg = Path("../CA_data/ca_districtr_bg_view_v1.gpkg")
layers = fiona.listlayers(gpkg)

# You should see a layer that says something like "ca_districtr_county_view_v1"
print(layers)

gdf = gpd.read_file(gpkg, layer="ca_districtr_bg_view_v1")
# Add coordinates for plotting
gdf['C_X'] = gdf.centroid.x
gdf['C_Y'] = gdf.centroid.y
print(gdf.head())


# Convert geopackage first into a gpd graph
gerry_graph = Graph.from_geodataframe(gdf)

# Check number of nodes (block groups)
print(len(gerry_graph.nodes()))

# Check node
print(gerry_graph.nodes()[1])

## Visualize the Block Groups

In [ ]:
import networkx as nx

# Draws graph with nodes
nx.draw(gerry_graph,pos = {x:(gerry_graph.nodes[x]['C_X'],gerry_graph.nodes[x]['C_Y']) for x in gerry_graph.nodes()}, node_size=20, cmap='coolwarm_r')
######node_color=[gerry_graph.nodes[x]['P0010003']/(gerry_graph.nodes[x]['P0010001']+1) for x in gerry_graph.nodes()],

# Draw map
gdf.plot()

## Deal with islands
We identify islands and disconnected components in the graph and manually connect them to geographically closest nodes.
Documentation: 
https://github.com/mggg/maup/wiki/What-to-do-about-islands-and-connectivity

In [24]:
# First check if islands or small graph compnents exist

import networkx as nx

# gerry_graph is gpd graph
islands = gerry_graph.islands

# print node numbers that are islands
print(islands)

# Look at graph components and the length of each
components = list(nx.connected_components(gerry_graph))
print([len(c) for c in components])

set()
[25607]


In [ ]:
# Identify the nodes that are not in largest component, print node numbers, visualize on graph

biggest_component_size = max(len(c) for c in components)
problem_components = [c for c in components if len(c) != biggest_component_size]
problem_nodes = [node for component in problem_components for node in component]
print(problem_nodes)

# Create a color column: red for problem node rows, lightgray otherwise
gdf['color'] = ['red' if i in problem_nodes else 'lightgray' for i in gdf.index]

# Plot with custom colors
gdf.plot(color=gdf['color'])


[]


In [ ]:
# For each problem node, find 5 nodes that have closest coordinates

import geopandas as gpd
import pandas as pd

# Reproject to a CRS with meters as units (if not already)
gdf = gdf.to_crs("EPSG:3857")

# Exclude all highlighted rows from possible neighbors
neighbor_candidates = gdf.drop(index=problem_nodes)

# Dictionary to hold 5 nearest neighbors for each highlighted row
nearest_rows = {}

for i in problem_nodes:
    source_geom = gdf.loc[i].geometry

    # Calculate distances to all non-highlighted rows
    distances = neighbor_candidates.geometry.distance(source_geom)

    # Get the indices of the 5 closest geometries
    nearest_indices = distances.nsmallest(5).index.tolist()

    nearest_rows[i] = nearest_indices

# Show the results
for src, neighbors in nearest_rows.items():
    print(f"Row {src} → Nearest rows: {neighbors}")

In [ ]:
# Visualize just the problem nodes and their close neighbors

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Collect all rows to visualize
all_rows = set(problem_nodes)
for neighbors in nearest_rows.values():
    all_rows.update(neighbors)

# Subset the GeoDataFrame
subset_gdf = gdf.loc[list(all_rows)].copy()

# Create a color map for the highlights
n_highlights = len(problem_nodes)
colormap = cm.get_cmap('tab10', n_highlights)  # You can also try 'Set3', 'Dark2', etc.
highlight_color_map = {idx: mcolors.to_hex(colormap(i)) for i, idx in enumerate(problem_nodes)}

# Assign colors
def assign_color(idx):
    if idx in problem_nodes:
        return highlight_color_map[idx]
    else:
        return 'lightgray'  # Default color for neighbors

subset_gdf['color'] = subset_gdf.index.map(assign_color)

# Plot
fig, ax = plt.subplots(figsize=(30, 20))
subset_gdf.plot(ax=ax, color=subset_gdf['color'], edgecolor='black')

# Label each shape with its row index
for idx, row in subset_gdf.iterrows():
    point = row.geometry.representative_point()
    ax.text(point.x, point.y, str(idx), fontsize=10, ha='center', va='center', color='black')

# Title and styling
ax.set_title("Unique Colors for Highlighted Rows + Neighbors", fontsize=24)
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# Manually add edges for disconnected nodes

from itertools import combinations

# Add nodes
gerry_graph.add_edge(20398, 10847)
gerry_graph.add_edge(20398, 20402)

# Node groups
n1 = [9431, 25412, 25413, 9427, 9428, 9429, 9430, 9432]
n2 = [9427, 9428, 9429, 9430, 9432, 10568, 13650, 6631, 6636, 6626, 6629]

# Add edges between every pair of nodes
gerry_graph.add_edges_from(combinations(n1, 2))
gerry_graph.add_edges_from(combinations(n2, 2))


# Check last time to see if islands have been connected
print(gerry_graph.islands)

## Save graph to json


In [26]:
# Convert gpd graph into json. This saves json file in directory.
gerry_graph.to_json("ca_districtr_bg_view_v1.json")

## Test run chain

In [ ]:
from gerrychain_cli import run_chain
from pathlib import Path


def run_single_chain():
    graph_file = Path("ca_districtr_bg_view_v1.json").resolve()

    output_dir = Path("./chain_out").resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    run_chain(
        graph_file=str(graph_file),
        n_districts=8,
        n_iterations=1,
        population_attr='total_vap_20',
        epsilon=0.00001,
        rng_seed=42,
        output_file=str(output_dir / "test_single.json"),
    )


run_single_chain()

# if __name__ == "__main__":
#     run_single_chain()
